In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# ==========================================================
# PARAMETERS
# ==========================================================

DB = "customer_health"
SILVER = "SILVER"

spark.conf.set("spark.sql.session.timeZone", "UTC")

try:

    # ==========================================================
    # CREATE TARGET TABLES
    # ==========================================================

    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB}.{SILVER}.CUSTOM_ACTIVITIES_ALL_LEADS_DETAILS
    (
        lead_id STRING,
        activity_id STRING,
        activity_at TIMESTAMP,
        wrapped_activities ARRAY<MAP<STRING,STRING>>,
        insert_date TIMESTAMP,
        update_date TIMESTAMP,
        md5_hash STRING
    )
    USING DELTA
    """)

    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB}.{SILVER}.LEADS_ACTIVITIES_SUMMARY
    (
        activity_id STRING,
        lead_id STRING,
        contact_id STRING,
        user_id STRING,
        type STRING,
        text STRING,
        status_change STRING,
        created_by_name STRING,
        updated_by_name STRING,
        attendees STRING,
        custom_activity_id STRING,
        meeting_starts_at TIMESTAMP,
        meeting_end_at TIMESTAMP,
        meeting_status STRING,
        activity_at TIMESTAMP,
        date_created TIMESTAMP,
        date_updated TIMESTAMP,
        created_by_user STRING,
        updated_by_user STRING,
        insert_date TIMESTAMP,
        update_date TIMESTAMP,
        wrapped_activities ARRAY<MAP<STRING,STRING>>,
        custom_activity STRING,
        custom_activity_outcome_name STRING,
        custom_activity_outcome STRING,
        dea_internal_name STRING,
        dea_internal_email STRING
    )
    USING DELTA
    """)

    # ==========================================================
    # LOAD ACTIVITY JSON
    # ==========================================================

    transient_df = spark.table(
        f"{DB}.{SILVER}.LEAD_ACTIVITIES_PROCESSED_TRANSIENT"
    )

    parsed_df = (
        transient_df
        .withColumn(
            "activity_json",
            F.expr(
                "customer_health.silver.PARSE_ACTIVITY_FINAL(json_object)"
            )
        )
        .withColumn(
            "activities",
            F.from_json(
                F.col("activity_json"),
                """
                array<
                    map<string,string>
                >
                """
            )
        )
    )

    # ==========================================================
    # EXPLODE ACTIVITIES
    # ==========================================================

    activities_df = (
        parsed_df
        .withColumn(
            "activity",
            F.explode("activities")
        )
    )

    # ==========================================================
    # CUSTOM ACTIVITY RECORDS
    # ==========================================================

    custom_activity_df = (
        activities_df
        .filter(
            F.col("activity")["_type"] == "CustomActivity"
        )
        .select(
            F.col("activity")["lead_id"].alias("lead_id"),
            F.col("activity")["id"].alias("activity_id"),
            F.to_timestamp(
                F.col("activity")["activity_at"]
            ).alias("activity_at"),
            F.col("activity")["custom_activity_type_id"]
                .alias("custom_activity_type_id"),
            "activity"
        )
    )

    # ==========================================================
    # EXPLODE KEYS
    # ==========================================================

    custom_details_df = (
        custom_activity_df
        .withColumn(
            "custom_activity_detail",
            F.explode(
                F.map_keys("activity")
            )
        )
        .filter(
            F.col("custom_activity_detail")
            .like("custom.%")
        )
        .select(
            "*",
            F.col("activity")[
                F.col("custom_activity_detail")
            ].alias(
                "custom_activity_detail_value"
            )
        )
    )

    # ==========================================================
    # JOIN CUSTOM ACTIVITIES
    # ==========================================================

    custom_lookup = spark.table(
        f"{DB}.{SILVER}.CUSTOM_ACTIVITIES"
    )

    final_ca = (
        custom_details_df.alias("d")
        .join(
            custom_lookup.alias("c"),
            (
                F.regexp_replace(
                    F.col(
                        "d.custom_activity_type_id"
                    ),
                    "custom\\.",
                    ""
                )
                ==
                F.col(
                    "c.custom_activity_id"
                )
            )
            &
            (
                F.regexp_replace(
                    F.col(
                        "d.custom_activity_detail"
                    ),
                    "custom\\.",
                    ""
                )
                ==
                F.col(
                    "c.custom_activity_outcome_id"
                )
            ),
            "inner"
        )
        .select(
            "lead_id",
            "activity_id",
            "activity_at",
            "custom_activity_type_id",
            "custom_activity_detail",
            "custom_activity_detail_value",
            "custom_activity_name",
            "custom_activity_outcome_name"
        )
    )

    # ==========================================================
    # WRAP ACTIVITIES
    # ==========================================================

    wrapped_df = (
        final_ca
        .groupBy(
            "lead_id",
            "activity_id",
            "activity_at"
        )
        .agg(
            F.collect_list(
                F.create_map(
                    F.lit(
                        "CUSTOM_ACTIVITY_TYPE_ID"
                    ),
                    F.col(
                        "custom_activity_type_id"
                    ),
                    F.lit(
                        "CUSTOM_ACTIVITY_DETAIL"
                    ),
                    F.col(
                        "custom_activity_detail"
                    ),
                    F.lit(
                        "CUSTOM_ACTIVITY_NAME"
                    ),
                    F.col(
                        "custom_activity_name"
                    ),
                    F.lit(
                        "CUSTOM_ACTIVITY_OUTCOME_NAME"
                    ),
                    F.col(
                        "custom_activity_outcome_name"
                    ),
                    F.lit(
                        "CUSTOM_ACTIVITY_DETAIL_VALUE"
                    ),
                    F.col(
                        "custom_activity_detail_value"
                    )
                )
            ).alias(
                "wrapped_activities"
            )
        )
        .withColumn(
            "insert_date",
            F.current_timestamp()
        )
        .withColumn(
            "update_date",
            F.current_timestamp()
        )
        .withColumn(
            "md5_hash",
            F.md5(
                F.concat_ws(
                    "||",
                    F.coalesce(
                        F.col("lead_id"),
                        F.lit("")
                    ),
                    F.coalesce(
                        F.col("activity_id"),
                        F.lit("")
                    ),
                    F.coalesce(
                        F.col("activity_at")
                        .cast("string"),
                        F.lit("")
                    ),
                    F.coalesce(
                        F.to_json(
                            F.col(
                                "wrapped_activities"
                            )
                        ),
                        F.lit("")
                    )
                )
            )
        )
    )

    # ==========================================================
    # MERGE CUSTOM_ACTIVITIES_ALL_LEADS_DETAILS
    # ==========================================================

    target = DeltaTable.forName(
        spark,
        f"{DB}.{SILVER}.CUSTOM_ACTIVITIES_ALL_LEADS_DETAILS"
    )

    (
        target.alias("tgt")
        .merge(
            wrapped_df.alias("src"),
            """
            tgt.lead_id = src.lead_id
            AND tgt.activity_id = src.activity_id
            AND tgt.activity_at = src.activity_at
            """
        )
        .whenMatchedUpdate(
            set={
                "wrapped_activities":
                    "src.wrapped_activities",
                "update_date":
                    "current_timestamp()",
                "md5_hash":
                    "src.md5_hash"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    # ==========================================================
    # LOAD PROCESSED ACTIVITIES
    # ==========================================================

    activities_processed = spark.table(
        f"{DB}.{SILVER}.LEAD_ACTIVITIES_PROCESSED"
    )

    users = spark.table(
        f"{DB}.{SILVER}.CLOSE_CRM_USERS_PROCESSED"
    )

    # ==========================================================
    # JOIN WRAPPED ACTIVITIES
    # ==========================================================

    summary_base = (
        activities_processed.alias("a")
        .join(
            wrapped_df.alias("w"),
            [
                "lead_id",
                "activity_id"
            ],
            "left"
        )
    )

    # ==========================================================
    # EXPLODE WRAPPED ACTIVITIES
    # ==========================================================

    summary_df = (
        summary_base
        .withColumn(
            "wrapped_activity",
            F.explode_outer(
                "wrapped_activities"
            )
        )
        .join(
            users.alias("u"),
            F.col("a.user_id")
            ==
            F.col("u.user_id"),
            "left"
        )
        .select(
            "a.*",
            "wrapped_activities",
            F.col(
                "wrapped_activity"
            )[
                "CUSTOM_ACTIVITY_NAME"
            ].alias(
                "custom_activity"
            ),
            F.col(
                "wrapped_activity"
            )[
                "CUSTOM_ACTIVITY_OUTCOME_NAME"
            ].alias(
                "custom_activity_outcome_name"
            ),
            F.col(
                "wrapped_activity"
            )[
                "CUSTOM_ACTIVITY_DETAIL_VALUE"
            ].alias(
                "custom_activity_outcome"
            ),
            F.concat_ws(
                " ",
                F.col(
                    "u.first_name"
                ),
                F.col(
                    "u.last_name"
                )
            ).alias(
                "dea_internal_name"
            ),
            F.col(
                "u.email"
            ).alias(
                "dea_internal_email"
            )
        )
    )

    # ==========================================================
    # REBUILD SUMMARY TABLE
    # ==========================================================

    summary_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option(
            "overwriteSchema",
            "true"
        ) \
        .saveAsTable(
            f"{DB}.{SILVER}.LEADS_ACTIVITIES_SUMMARY"
        )

    print(
        "Stored Procedure Executed Successfully"
    )

except Exception as e:

    raise Exception(
        f"Error: {str(e)}"
    )